# parity

> Compare the MAX graph and the Kaldi frontend with PyTorch and torchaudio.
>
> The log-mel is checked first. A tiny ViT is then compiled and compared
> token for token. If `models/audio_jepa.safetensors` exists, the real
> checkpoint is compared too.


In [ ]:
#| default_exp parity


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np

from nada.config import AudioJEPAConfig, pretrained_config, tiny_config
from nada.features import fbank, prepare
from nada.model import build, compile_model, load_checkpoint, to_device
from nada.weights import assign_parameters


In [ ]:
#| export
def main() -> None:
    check_fbank()
    check_model()
    check_pretrained()
    print("parity ok")


In [ ]:
#| export
def check_fbank() -> None:
    import torch
    import torchaudio

    config = pretrained_config()
    clip = np.random.default_rng(0).standard_normal(config.sample_rate).astype(np.float32) * 0.1
    waveform = torch.from_numpy(clip)[None]
    waveform = waveform - waveform.mean()
    reference = torchaudio.compliance.kaldi.fbank(
        waveform,
        sample_frequency=config.sample_rate,
        frame_length=config.frame_length_ms,
        frame_shift=config.hop_length_ms,
        num_mel_bins=config.n_mels,
        low_freq=config.f_min,
        high_freq=config.f_max,
        use_log_fbank=True,
        window_type="hanning",
    ).numpy()
    got = fbank(clip, config)
    _compare("fbank", got, reference, 1e-4)


In [ ]:
#| export
def _tiny_reference(config: AudioJEPAConfig):
    import torch
    from torch import nn

    class Attention(nn.Module):
        def __init__(self, dim, heads, qkv_bias=True):
            super().__init__()
            self.heads = heads
            self.head_dim = dim // heads
            self.scale = self.head_dim ** -0.5
            self.qkv = nn.Linear(dim, 3 * dim, bias=qkv_bias)
            self.proj = nn.Linear(dim, dim)

        def forward(self, x):
            b, n, _ = x.shape
            qkv = self.qkv(x).reshape(b, n, 3, self.heads, self.head_dim)
            q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
            context = (q @ k.transpose(-2, -1) * self.scale).softmax(-1) @ v
            return self.proj(context.transpose(1, 2).reshape(b, n, self.heads * self.head_dim))

    class Block(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.norm1 = nn.LayerNorm(config.embed_dim, eps=config.layer_norm_eps)
            self.attn = Attention(config.embed_dim, config.num_heads, config.qkv_bias)
            self.norm2 = nn.LayerNorm(config.embed_dim, eps=config.layer_norm_eps)
            class MLP(nn.Module):
                def __init__(self, dim, hidden):
                    super().__init__()
                    self.fc1 = nn.Linear(dim, hidden)
                    self.fc2 = nn.Linear(hidden, dim)

                def forward(self, x):
                    return self.fc2(nn.functional.gelu(self.fc1(x)))

            self.mlp = MLP(config.embed_dim, config.mlp_hidden)

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))

    class Encoder(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.patch_embed = nn.Module()
            self.patch_embed.proj = nn.Conv2d(
                config.in_chans, config.embed_dim, kernel_size=config.patch_size, stride=config.patch_size
            )
            self.pos_embed = nn.Parameter(torch.zeros(1, config.num_patches, config.embed_dim))
            self.blocks = nn.ModuleList([Block(config) for _ in range(config.depth)])
            self.norm = nn.LayerNorm(config.embed_dim, eps=config.layer_norm_eps)

        def forward(self, x):
            x = self.patch_embed.proj(x).flatten(2).transpose(1, 2) + self.pos_embed
            for block in self.blocks:
                x = block(x)
            return self.norm(x)

    reference = Encoder(config)
    generator = torch.Generator().manual_seed(0)
    with torch.no_grad():
        for parameter in reference.parameters():
            parameter.add_(0.2 * torch.randn(parameter.shape, generator=generator))
    return reference.eval()


In [ ]:
#| export
def _compare(name: str, got, expected, atol: float) -> None:
    got = np.asarray(got)
    delta = float(np.max(np.abs(got - expected)))
    print(f"{name} max abs {delta:.3e}")
    np.testing.assert_allclose(got, expected, atol=atol, rtol=atol)


In [ ]:
#| export
def check_model() -> None:
    import torch

    config = tiny_config()
    reference = _tiny_reference(config)
    model = build(config)
    arrays = {key: value.detach().cpu().numpy() for key, value in reference.state_dict().items()}
    assign_parameters(model, arrays)

    spectrogram = np.random.default_rng(1).standard_normal(
        (2, config.time_bins, config.n_mels, config.in_chans)
    ).astype(np.float32)
    nchw = np.transpose(spectrogram, (0, 3, 1, 2))
    with torch.no_grad():
        expected = reference(torch.from_numpy(nchw)).numpy()
    compiled = compile_model(model, batch=2)
    tokens = compiled(to_device(model, spectrogram))
    _compare("tokens", tokens.to_numpy(), expected, 1e-4)


In [ ]:
#| export
def check_pretrained() -> None:
    import torch

    from nada.core import repo_root

    checkpoint = repo_root() / "models" / "audio_jepa.safetensors"
    if not checkpoint.is_file():
        print("pretrained skipped: run `pixi run -e reference convert` to compare the real checkpoint")
        return
    from nada.weights import load_arrays

    config = pretrained_config()
    arrays = load_arrays(checkpoint)
    reference = _tiny_reference(config)
    reference.load_state_dict({key: torch.from_numpy(value) for key, value in arrays.items()}, strict=True)
    reference.eval()
    model = build(config)
    load_checkpoint(model, checkpoint)

    time = np.arange(config.sample_rate, dtype=np.float64) / config.sample_rate
    clip = (0.3 * np.sin(2 * np.pi * 220 * time)).astype(np.float32)
    values = prepare([clip], config)
    nchw = np.transpose(values, (0, 3, 1, 2))
    with torch.no_grad():
        expected = reference(torch.from_numpy(nchw)).numpy()
    compiled = compile_model(model, batch=1)
    tokens = compiled(to_device(model, values))
    _compare("pretrained tokens", tokens.to_numpy(), expected, 5e-3)


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    main()
